In [1]:
import pandas as pd

In [2]:
pr_df = pd.read_parquet("hf://datasets/hao-li/AIDev/pull_request.parquet")

In [3]:
from scipy.stats import spearmanr

pr_commit_details_df = pd.read_parquet("hf://datasets/hao-li/AIDev/pr_commit_details.parquet")

def prep_data(pr_df, pr_commit_details_df):
    agg = pr_commit_details_df.groupby('pr_id').agg(
        additions=('additions', 'sum'),
        deletions=('deletions', 'sum'),
        commits=('sha', 'nunique'),
    ).reset_index()

    df = pr_df.merge(agg, left_on='id', right_on='pr_id', how='left')
    df['pr_size'] = df['additions'].fillna(0) + df['deletions'].fillna(0)
    df['commits_safe'] = df['commits'].fillna(1).replace(0, 1)
    df['commit_granularity'] = df['pr_size'] / df['commits_safe']
    df['is_merged'] = df['merged_at'].notna().astype(int)
    return df

def analyze_correlation(df, feature_col, target_col):
    clean_df = df.dropna(subset=[feature_col, target_col])
    coef, p_val = spearmanr(clean_df[feature_col], clean_df[target_col])
    return coef, p_val

df = prep_data(pr_df, pr_commit_details_df)

corr_size, p_size = analyze_correlation(df, 'pr_size', 'is_merged')
corr_gran, p_gran = analyze_correlation(df, 'commit_granularity', 'is_merged')

print(f"Spearman Correlation (PR Size vs Merged): {corr_size:.4f} (p-value: {p_size:.4e})")
print(f"Spearman Correlation (Granularity vs Merged): {corr_gran:.4f} (p-value: {p_gran:.4e})")

Spearman Correlation (PR Size vs Merged): -0.1386 (p-value: 1.2127e-143)
Spearman Correlation (Granularity vs Merged): -0.0682 (p-value: 6.1537e-36)
